In [4]:
import requests
import pandas as pd

# -----------------------
# Credentials
# -----------------------
token_url = "https://app1pub.smappee.net/dev/v3/oauth2/token"

client_id = "165804"
client_secret = "ty2HNwm2AF"
username = "Ilias.EsSahili@oracdecor.be"
password = "Traverse4-Yonder-Skylight-Unsubtle-Epidermal!"

# -----------------------
# Get Access Token
# -----------------------
payload = {
    "grant_type": "password",
    "client_id": client_id,
    "client_secret": client_secret,
    "username": username,
    "password": password
}

headers = {
    "Content-Type": "application/x-www-form-urlencoded;charset=UTF-8"
}

token_response = requests.post(token_url, data=payload, headers=headers)
token_response.raise_for_status()

access_token = token_response.json()["access_token"]

In [5]:
print("Access Token:", access_token)

Access Token: d8129dcc-1d58-3759-86e0-1c7a9380485d


In [6]:
import json

url = "https://app1pub.smappee.net/dev/v3/servicelocation"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

servicelocations = data["serviceLocations"]

print(json.dumps(servicelocations, indent=4))



[
    {
        "serviceLocationId": 75452,
        "serviceLocationUuid": "01d1d4a4-32b0-4be4-90ee-a0a78920f542",
        "name": "Orac",
        "deviceSerialNumber": "5010005672"
    },
    {
        "serviceLocationId": 80060,
        "serviceLocationUuid": "34666732-62c6-465d-ada6-27fdd30c5ab4",
        "name": "Orac Oostende laadplein",
        "deviceSerialNumber": "5130099474"
    },
    {
        "serviceLocationId": 80071,
        "serviceLocationUuid": "903908ae-172d-493c-956a-5f3e58fbdbb3",
        "name": "Ev Base Orac 1",
        "deviceSerialNumber": "5130005802"
    },
    {
        "serviceLocationId": 80072,
        "serviceLocationUuid": "9200f38f-4f99-4db2-a074-acbd2cd7ca44",
        "name": "Ev Base Orac 2",
        "deviceSerialNumber": "5130008024"
    },
    {
        "serviceLocationId": 80074,
        "serviceLocationUuid": "38b908ae-699f-49fa-bea8-9df869d65a80",
        "name": "Ev Base Orac 3",
        "deviceSerialNumber": "5130008032"
    },
    {
        

In [ ]:
import requests

def get_consumption(service_location_id, access_token, aggregation, from_ts, to_ts):

    url = f"https://app1pub.smappee.net/dev/v3/servicelocation/{service_location_id}/consumption"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    params = {
        "aggregation": aggregation,
        "from": int(from_ts),
        "to": int(to_ts)
    }

    response = requests.get(url, headers=headers, params=params)

    print("REQUEST URL:", response.url)  # 🔥 DEBUG IMPORTANT
    print("STATUS:", response.status_code)
    print("RAW RESPONSE:", response.text[:500])

    response.raise_for_status()

    return response.json()

In [ ]:
import requests

def get_servicelocation_info(service_location_id, access_token):

    url = f"https://app1pub.smappee.net/dev/v3/servicelocation/{service_location_id}/info"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    # Geen params nodig (zoals from_ts of to_ts) voor de info endpoint
    response = requests.get(url, headers=headers)

    print("REQUEST URL:", response.url) 
    print("STATUS:", response.status_code)
    # We printen hier iets meer tekens (1000) omdat de info-JSON vaak best groot is
    print("RAW RESPONSE:", response.text[:1000]) 

    response.raise_for_status()

    return response.json()

In [8]:
import time
import pandas as pd
import matplotlib.pyplot as plt


def get_time_range(period: str):
    """
    period: 'day', 'week', 'month', 'year'
    returns from_ts, to_ts in milliseconds
    """
    to_ts = int(time.time() * 1000)

    if period == "day":
        from_ts = to_ts - (1 * 24 * 60 * 60 * 1000)
    elif period == "week":
        from_ts = to_ts - (7 * 24 * 60 * 60 * 1000)
    elif period == "month":
        from_ts = to_ts - (30 * 24 * 60 * 60 * 1000)
    elif period == "year":
        from_ts = to_ts - (365 * 24 * 60 * 60 * 1000)
    else:
        raise ValueError("Invalid period. Use: day, week, month, year")

    return from_ts, to_ts


def plot_energy(data, period="week"):
    # timestamps bepalen (optioneel, mag weg als je het extern doet)
    from_ts, to_ts = get_time_range(period)

    # data ophalen
    rows = data["consumptions"]

    df = pd.DataFrame(rows)

    # timestamp → datetime
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.sort_values("date")

    # kWh conversie
    df["consumption_kwh"] = df["consumption"] / 1000
    df["solar_kwh"] = df["solar"] / 1000
    df["gridImport_kwh"] = df["gridImport"] / 1000
    df["gridExport_kwh"] = df["gridExport"] / 1000

    # plot
    plt.figure(figsize=(14, 6))

    plt.plot(df["date"], df["consumption_kwh"], label="Consumption (kWh)")
    plt.plot(df["date"], df["solar_kwh"], label="Solar (kWh)")
    plt.plot(df["date"], df["gridImport_kwh"], linestyle="dotted", label="Grid Import (kWh)")
    plt.plot(df["date"], df["gridExport_kwh"], linestyle="dotted", label="Grid Export (kWh)")

    plt.title(f"Energy overview ({period})")
    plt.xlabel("Date")
    plt.ylabel("kWh")

    plt.legend()

    # 🔥 clean date formatting (fix van jouw issue)
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%m %H:%M'))
    plt.gcf().autofmt_xdate()

    plt.tight_layout()
    plt.show()

In [12]:
data = get_servicelocation_info(service_location_id=75452, access_token=access_token)

print(data)

REQUEST URL: https://app1pub.smappee.net/dev/v3/servicelocation/75452/info
STATUS: 200
RAW RESPONSE: {
  "lon": 2.9612479567454173,
  "lat": 51.218414306640625,
  "electricityCost": 0.0,
  "electricityCurrency": "EUR",
  "timezone": "Europe/Brussels",
  "appliances": [],
  "actuators": [],
  "sensors": [],
  "monitors": [],
  "channelsConfiguration": {
    "inputChannels": [
      {
        "ctInput": 0,
        "name": "Grid",
        "phase": 0,
        "reversed": false,
        "nilm": false,
        "balanced": false,
        "inputChannelCTType": "CT50_100_200"
      },
      {
        "ctInput": 0,
        "name": "Grid",
        "phase": 1,
        "reversed": false,
        "nilm": false,
        "balanced": false,
        "inputChannelCTType": "CT50_100_200"
      },
      {
        "ctInput": 0,
        "name": "Grid",
        "phase": 2,
        "reversed": false,
        "nilm": false,
        "balanced": false,
        "inputChannelCTType": "CT50_100_200"
      },
      {

In [17]:
from_ts, to_ts = get_time_range("day")

data = get_consumption(
    service_location_id=75452,
    access_token=access_token,
    aggregation=2,
    from_ts=from_ts,
    to_ts=to_ts
)

REQUEST URL: https://app1pub.smappee.net/dev/v3/servicelocation/75452/consumption?aggregation=2&from=1779717880101&to=1779804280101
STATUS: 200
RAW RESPONSE: {"serviceLocationId":75452,"consumptions":[{"timestamp":1779721200000,"consumption":355661.533,"solar":122763.68,"alwaysOn":31384.757,"gridImport":71.7,"gridExport":12700.809,"selfConsumption":89.65,"selfSufficiency":99.98,"active":[null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null],"reactive":[null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [18]:
print(data)

{'serviceLocationId': 75452, 'consumptions': [{'timestamp': 1779721200000, 'consumption': 355661.533, 'solar': 122763.68, 'alwaysOn': 31384.757, 'gridImport': 71.7, 'gridExport': 12700.809, 'selfConsumption': 89.65, 'selfSufficiency': 99.98, 'active': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'reactive': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'voltages': [None, None, None], 'phaseVoltages': [None, None, None], 'currentHarmonics': [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []], 'voltageHarmonics': [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]}, {'timestamp': 1779724800000, 'consumption': 1096898.177, '

In [ ]:
from_ts, to_ts = get_time_range("day")

data2 = get_consumption(
    service_location_id=80060,
    access_token=access_token,
    aggregation=7,
    from_ts=from_ts,
    to_ts=to_ts
)

print(data2)

plot_energy(data2, period="week")

NameError: name 'get_time_range' is not defined

In [ ]:
def transform_to_db_format(data, service_location_id):
    rows = data["consumptions"]

    df = pd.DataFrame(rows)

    # -----------------------
    # timestamp fix
    # -----------------------
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

    # -----------------------
    # kWh conversie
    # -----------------------
    df["consumption_kwh"] = df["consumption"] / 1000
    df["solar_kwh"] = df["solar"] / 1000
    df["grid_import_kwh"] = df["gridImport"] / 1000
    df["grid_export_kwh"] = df["gridExport"] / 1000

    # -----------------------
    # extra kolommen voor DB
    # -----------------------
    df["service_location_id"] = service_location_id
    df["timestamp"] = df["timestamp"]  # blijft bigint

    # -----------------------
    # alleen kolommen voor SQL tabel
    # -----------------------
    df = df[[
        "service_location_id",
        "timestamp",
        "date",
        "consumption_kwh",
        "solar_kwh",
        "grid_import_kwh",
        "grid_export_kwh"
    ]]

    return df

In [ ]:
df = pd.DataFrame(data2["consumptions"])

df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

df["consumption_kwh"] = df["consumption"] / 1000
df["solar_kwh"] = df["solar"] / 1000
df["grid_import_kwh"] = df["gridImport"] / 1000
df["grid_export_kwh"] = df["gridExport"] / 1000


In [ ]:
#Zorg dat alle tabellen waar waarden in staan die met een . gedecimalen zijn, naar een , decimaal omgezet worden.
df["consumption_kwh"] = df["consumption_kwh"].apply(lambda x: str(x).replace(".", ","))
df["solar_kwh"] = df["solar_kwh"].apply(lambda x: str(x).replace(".", ","))
df["grid_import_kwh"] = df["grid_import_kwh"].apply(lambda x: str(x).replace(".", ","))
df["grid_export_kwh"] = df["grid_export_kwh"].apply(lambda x: str(x).replace(".", ","))


In [ ]:

df.to_csv("energy_data_ev_hourly.csv", index=False)